In [1]:
import logging
from pyspark.sql import functions as F


def get_logger(name):
    """Create a configured logger for a given layer/table."""
    logging.basicConfig(level=logging.INFO)
    return logging.getLogger(name)

StatementMeta(, 005554b5-cb95-4ef9-944b-8d5ae79e65d8, 3, Finished, Available, Finished, False)

In [2]:
def run_dq_checks(checks, logger, table_name):
    """
    Run a list of DQ check dicts and raise if any failed.

    checks: list of dicts like {"name": str, "passed": bool, "message": str}
    logger: a logger instance from get_logger()
    table_name: used in the exception message if checks fail
    """
    dq_results = []
    for check in checks:
        status = "PASS" if check["passed"] else "FAIL"
        log_msg = f"[DQ] [{status}] {check['name']} — {check['message']}"
        (logger.info if check["passed"] else logger.error)(log_msg)
        dq_results.append({**check, "status": status})

    failed = [r for r in dq_results if r["status"] == "FAIL"]
    if failed:
        failed_names = ", ".join([r["name"] for r in failed])
        raise Exception(
            f"{table_name} DQ failed — {len(failed)} check(s) failed: "
            f"{failed_names}. Pipeline halted."
        )

    logger.info(f"[DQ] All checks passed for {table_name}.")

StatementMeta(, 005554b5-cb95-4ef9-944b-8d5ae79e65d8, 4, Finished, Available, Finished, False)

In [3]:
def check_row_count(df, expected_count):
    actual = df.count()
    return {
        "name": "Row count",
        "passed": actual == expected_count,
        "message": f"Expected {expected_count:,}, got {actual:,}"
    }

StatementMeta(, 005554b5-cb95-4ef9-944b-8d5ae79e65d8, 5, Finished, Available, Finished, False)

In [4]:
def check_null_pk(df, pk_col):
    null_count = df.filter(F.col(pk_col).isNull()).count()
    return {
        "name": f"Null PK — {pk_col}",
        "passed": null_count == 0,
        "message": f"{null_count:,} nulls"
    }

StatementMeta(, 005554b5-cb95-4ef9-944b-8d5ae79e65d8, 6, Finished, Available, Finished, False)

In [5]:
def check_duplicate_pk(df, pk_cols):
    if isinstance(pk_cols, str):
        pk_cols = [pk_cols]
    total = df.count()
    distinct = df.select(*pk_cols).distinct().count()
    dup_count = total - distinct
    return {
        "name": f"Duplicate PK — {', '.join(pk_cols)}",
        "passed": dup_count == 0,
        "message": f"{dup_count:,} duplicates"
    }

StatementMeta(, 005554b5-cb95-4ef9-944b-8d5ae79e65d8, 7, Finished, Available, Finished, False)

In [6]:
def check_referential_integrity(df_fact, df_dim, join_col, dim_join_col=None, dim_name="dimension"):
    """Checks that every non-null value in df_fact[join_col] exists in df_dim[dim_join_col]."""
    dim_join_col = dim_join_col or join_col
    orphaned = df_fact.filter(F.col(join_col).isNotNull()).join(
        df_dim.select(F.col(dim_join_col)),
        df_fact[join_col] == df_dim[dim_join_col],
        how="left_anti"
    ).count()
    return {
        "name": f"Referential integrity — {join_col} → {dim_name}",
        "passed": orphaned == 0,
        "message": f"{orphaned:,} rows with {join_col} not found in {dim_name}"
    }

StatementMeta(, 005554b5-cb95-4ef9-944b-8d5ae79e65d8, 8, Finished, Available, Finished, False)

In [7]:
def check_join_coverage(df, id_col, name_col, label=None):
    """Checks rows that have an FK populated but the joined name/attribute came back null."""
    label = label or f"{name_col} join coverage"
    unmatched = df.filter(
        F.col(id_col).isNotNull() & F.col(name_col).isNull()
    ).count()
    return {
        "name": label,
        "passed": unmatched == 0,
        "message": f"{unmatched:,} rows with {id_col} but no {name_col}"
    }

StatementMeta(, 005554b5-cb95-4ef9-944b-8d5ae79e65d8, 9, Finished, Available, Finished, False)

In [8]:
def check_null_columns(df, cols):
    """
    Checks multiple columns for nulls in ONE aggregation pass instead of
    one .count() per column. Returns a LIST of check dicts (one per column),
    meant to be splatted into a checks=[...] list with the * operator:
        checks = [check_row_count(...), *check_null_columns(df, cols), ...]
    """
    null_counts = df.agg(
        *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]
    ).collect()[0]
 
    return [
        {
            "name": f"Null check — {c}",
            "passed": null_counts[c] == 0,
            "message": f"{null_counts[c]:,} nulls found"
        }
        for c in cols
    ]


StatementMeta(, 005554b5-cb95-4ef9-944b-8d5ae79e65d8, 10, Finished, Available, Finished, False)